# 01 · Data profiling: NYC Yellow Taxi

Phase 0, Step 0.7. This notebook measures the problems that already exist in the real taxi data,
before any rule is written. The findings decide the rules in Phase 2, the thresholds in Phase 3,
and which problems need fault injection in Phase 6.

**Before running:** download data with `python src/data_download/download_data.py`.
By default the notebook reads `data/incoming/`. Set `DATA_DIR` below to `data/sample` for a quick run.

| Section | Question | Check category |
|---|---|---|
| 1 | How many rows per file and per day? | Volume |
| 2 | Do columns or types differ between files? | Schema |
| 3 | Which columns have NULLs, and do they come in groups? | Completeness |
| 4–7 | Ranges, timestamps, zones, totals | Validity, pattern, referential, consistency |
| 8 | Duplicates | Uniqueness |
| 9 | Distributions of distance and fare | Drift baseline |
| 10 | Summary of findings | All |

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px

# Find the repository root (the folder with pyproject.toml) and make src/ importable.
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(REPO_ROOT / "src"))

from profiling import taxi_profile as tp

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

## Settings

In [ ]:
DATA_DIR = Path(os.environ.get("DATAGUARD_PROFILE_DIR", REPO_ROOT / "data" / "incoming"))
FILE_GLOB = "yellow_tripdata_*.parquet"
ZONES_CSV = DATA_DIR / "taxi_zone_lookup.csv"

files = sorted(DATA_DIR.glob(FILE_GLOB))
assert files, f"No files matching {FILE_GLOB} in {DATA_DIR}: run download_data.py first"
assert ZONES_CSV.exists(), f"{ZONES_CSV} not found: run download_data.py first"

con = tp.connect()
print(f"{len(files)} file(s) in {DATA_DIR}")
for f in files:
    print(f"  {f.name}  {f.stat().st_size / 1024 / 1024:.1f} MB")

## 1 · Volume: rows per file and per day

Months have different lengths, so the volume check (Phase 3) compares **rows per day**, not raw row counts.

In [ ]:
volume = tp.rows_per_file(con, files)
print(f"Total rows: {volume['rows'].sum():,}")
volume

In [ ]:
fig = px.bar(volume, x="period", y="rows_per_day", title="Rows per day by file")
fig.show()

## 2 · Schema: columns and types per file

Any column whose exact name or type is not the same in every file is listed. A name that differs only
by case (for example `airport_fee` vs `Airport_fee`) shows up as two rows.

In [ ]:
schema = tp.schema_by_file(con, files)
differences = tp.schema_differences(schema)
print(f"{schema['column'].nunique()} distinct column names; {len(differences)} differ between files")
differences

In [ ]:
# Full schema of the first file, for reference
schema[schema["file"] == files[0].name][["column", "type"]].reset_index(drop=True)

## 3 · Completeness: NULL rate per column (%)

The second table checks whether the NULLs in `passenger_count` arrive together with NULLs in
`RatecodeID`, `store_and_fwd_flag`, `congestion_surcharge` and `airport_fee`.

In [ ]:
nulls = tp.null_rates(con, files)
nulls[nulls.drop(columns="column").max(axis=1) > 0].reset_index(drop=True)

In [ ]:
group = tp.null_group(con, files)
group["group_share_pct"] = (100 * group["whole_group_null"] / group["passenger_count_null"]).round(2)
group

## 4–7 · Validity, pattern, referential and consistency checks

One row per file and check. `pct` is the share of the file's rows that fail the check.
Zones 264/265 are valid IDs but mean *Unknown* / *Outside of NYC*, so they are counted separately.

In [ ]:
issues = tp.issue_counts(con, files, ZONES_CSV)
issues.pivot(index="check", columns="file", values="pct")

In [ ]:
issues.pivot(index="check", columns="file", values="rows")

## 8 · Uniqueness: duplicates

`exact_duplicates`: rows identical in every column. `key_duplicates`: rows sharing the composite key
(vendor, pickup and dropoff time, locations, fare, total).

In [ ]:
tp.duplicate_counts(con, files)

## 9 · Distributions (drift baseline)

Quantiles per file. Phase 3 compares new files with these to detect drift, for example a unit change.

In [ ]:
dist = tp.distribution(con, files)
dist

In [ ]:
fig = px.line(
    dist,
    x="file",
    y="p50",
    color="column",
    markers=True,
    title="Median per file",
)
fig.show()

## 10 · Summary of findings

Every check across all files, most frequent first. This table feeds `docs/data_profile.md` (Step 0.8).

In [ ]:
findings = tp.summary(issues)
findings

In [ ]:
con.close()